In [1]:
#Retrieval-Augmented Recommendation


In [2]:
!pip install rank-bm25 pandas

In [1]:
# Sample Steam-style dataset

from rank_bm25 import BM25Okapi

# --------------------------------------------------
# Sample Steam-style dataset
# --------------------------------------------------

games = [
    {
        "id": 1,
        "title": "Euro Truck Simulator 2",
        "genres": "Simulation Driving",
        "tags": "Relaxing Open World Realistic",
        "description": "Drive trucks across Europe and deliver cargo."
    },
    {
        "id": 2,
        "title": "American Truck Simulator",
        "genres": "Simulation Driving",
        "tags": "Relaxing Open World Road Trip",
        "description": "Travel across the United States in a truck."
    },
    {
        "id": 3,
        "title": "Counter-Strike 2",
        "genres": "FPS Action",
        "tags": "Shooter Competitive Multiplayer",
        "description": "A tactical first person shooter."
    },
    {
        "id": 4,
        "title": "Stardew Valley",
        "genres": "Simulation RPG",
        "tags": "Farming Relaxing Pixel",
        "description": "Build your farm and enjoy a peaceful rural life."
    },
    {
        "id": 5,
        "title": "Coral Island",
        "genres": "Simulation RPG",
        "tags": "Farming Relaxing Cozy",
        "description": "Farm, craft, and explore a tropical island."
    },
    {
        "id": 6,
        "title": "Apex Legends",
        "genres": "FPS Action",
        "tags": "Shooter Battle Royale Multiplayer",
        "description": "Squad based battle royale."
    },
    {
        "id": 7,
        "title": "SnowRunner",
        "genres": "Simulation Driving",
        "tags": "Offroad Vehicles Realistic",
        "description": "Drive powerful vehicles through difficult terrain."
    }
]

# --------------------------------------------------
# Build BM25 corpus
# --------------------------------------------------

documents = []

for game in games:
    text = " ".join([
        game["title"],
        game["genres"],
        game["tags"],
        game["description"]
    ])
    documents.append(text.lower().split())

bm25 = BM25Okapi(documents)

# --------------------------------------------------
# Free text search
# --------------------------------------------------

def search(query, top_k=5):
    tokens = query.lower().split()

    scores = bm25.get_scores(tokens)

    ranked = sorted(
        zip(games, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_k]

# --------------------------------------------------
# Item-to-item recommendation
# --------------------------------------------------

def recommend_similar(game_id, top_k=5):

    target = next(g for g in games if g["id"] == game_id)

    query = (
        target["genres"]
        + " "
        + target["tags"]
        + " "
        + target["description"]
    )

    results = search(query, top_k + 1)

    filtered = [
        (g, s)
        for g, s in results
        if g["id"] != game_id
    ]

    return filtered[:top_k]


# --------------------------------------------------
# Demo 1
# --------------------------------------------------

print("\n=== TEXT QUERY ===\n")

results = search("relaxing driving simulation")

for rank, (game, score) in enumerate(results, start=1):
    print(
        f"{rank}. {game['title']:<30} score={score:.3f}"
    )

# --------------------------------------------------
# Demo 2
# --------------------------------------------------

print("\n=== SIMILAR TO ETS2 ===\n")

results = recommend_similar(1)

for rank, (game, score) in enumerate(results, start=1):
    print(
        f"{rank}. {game['title']:<30} score={score:.3f}"
    )


=== TEXT QUERY ===

1. Euro Truck Simulator 2         score=0.797
2. American Truck Simulator       score=0.774
3. Coral Island                   score=0.617
4. SnowRunner                     score=0.601
5. Stardew Valley                 score=0.581

=== SIMILAR TO ETS2 ===

1. American Truck Simulator       score=2.903
2. SnowRunner                     score=2.308
3. Coral Island                   score=0.872
4. Stardew Valley                 score=0.820
5. Counter-Strike 2               score=0.000


In [2]:
#hybrid?

#retrieve_similar_item_bm25()
#retrieve_similar_item_vector()
#retrieve_similar_item_hybrid()

#retrieve_query_bm25()
#retrieve_query_vector()
#retrieve_query_hybrid()

In [2]:
from rank_bm25 import BM25Okapi

In [3]:
# Demo on the 30 counts of sample data
import json
import re
from pathlib import Path

from rank_bm25 import BM25Okapi


# ============================================================
# CONFIG
# ============================================================

DATA_DIR = Path(
    "/Users/kesiyun/Desktop/5481 Data/5481Project Data/Steam_data"
)

ITEMS_FILE = DATA_DIR / "items_sample.jsonl"
TAGS_FILE = DATA_DIR / "tags_sample.jsonl"


# ============================================================
# HELPERS
# ============================================================

def tokenize(text: str):
    """
    Better than split()
    Keeps only word tokens.
    """

    return re.findall(r"\w+", text.lower())


def load_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))

    return rows


# ============================================================
# LOAD DATA
# ============================================================

items = load_jsonl(ITEMS_FILE)
annotations = load_jsonl(TAGS_FILE)

annotation_map = {
    row["item_id"]: row
    for row in annotations
}

print(f"Loaded items: {len(items)}")
print(f"Loaded annotations: {len(annotation_map)}")


# ============================================================
# BUILD SEARCH DOCUMENTS
# ============================================================

documents = []

for item in items:

    item_id = item["item_id"]

    anno = annotation_map.get(item_id, {})

    summary = anno.get("summary", "")

    llm_tags = []

    if "tags" in anno:

        for dimension, values in anno["tags"].items():

            if isinstance(values, list):
                llm_tags.extend(values)

    searchable_text = " ".join([
        item.get("name", ""),
        " ".join(item.get("genres", [])),
        " ".join(item.get("native_tags", [])),
        summary,
        " ".join(llm_tags),
        item.get("description_clean", "")
    ])

    documents.append(tokenize(searchable_text))


# ============================================================
# BM25 INDEX
# ============================================================

bm25 = BM25Okapi(documents)


# ============================================================
# QUERY SEARCH
# ============================================================

def search(query, top_k=10):

    tokens = tokenize(query)

    scores = bm25.get_scores(tokens)

    ranked = sorted(
        zip(items, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_k]


# ============================================================
# ITEM-TO-ITEM SEARCH
# ============================================================

def recommend_similar(item_id, top_k=10):

    target = next(
        item
        for item in items
        if item["item_id"] == item_id
    )

    anno = annotation_map.get(item_id, {})

    summary = anno.get("summary", "")

    query_text = " ".join([
        target.get("name", ""),
        " ".join(target.get("genres", [])),
        " ".join(target.get("native_tags", [])),
        summary
    ])

    results = search(query_text, top_k + 1)

    filtered = [
        (game, score)
        for game, score in results
        if game["item_id"] != item_id
    ]

    return filtered[:top_k]


# ============================================================
# PRETTY PRINT
# ============================================================

def print_result(game, score):

    item_id = game["item_id"]

    anno = annotation_map.get(item_id, {})

    tags = anno.get("tags", {})

    print("-" * 60)

    print(
        f"{game['name']}"
        f"  | score={score:.3f}"
    )

    print(
        f"Genres: {', '.join(game.get('genres', []))}"
    )

    print(
        f"Native tags: "
        f"{', '.join(game.get('native_tags', [])[:5])}"
    )

    if tags:

        mood = ", ".join(
            tags.get("mood", [])
        )

        theme = ", ".join(
            tags.get("theme", [])
        )

        print(f"Mood: {mood}")
        print(f"Theme: {theme}")

    summary = anno.get("summary", "")

    if summary:
        print(
            f"Summary: {summary[:120]}"
        )


# ============================================================
# DEMO 1
# ============================================================

print("\n")
print("=" * 60)
print("FREE TEXT SEARCH")
print("=" * 60)

query = "relaxing farming game"

results = search(query)

print(f"\nQuery: {query}\n")

for rank, (game, score) in enumerate(
    results,
    start=1
):
    print(f"\n#{rank}")
    print_result(game, score)


# ============================================================
# DEMO 2
# ============================================================

example_id = items[0]["item_id"]

print("\n")
print("=" * 60)
print("SIMILAR GAMES")
print("=" * 60)

print(
    f"\nTarget: "
    f"{items[0]['name']} "
    f"(id={example_id})"
)

results = recommend_similar(example_id)

for rank, (game, score) in enumerate(
    results,
    start=1
):
    print(f"\n#{rank}")
    print_result(game, score)

Loaded items: 30
Loaded annotations: 30


FREE TEXT SEARCH

Query: relaxing farming game


#1
------------------------------------------------------------
Lightyear Frontier  | score=5.164
Genres: Early Access
Native tags: Early Access, Farming Sim, Open World Survival Craft, Base Building, Automation
Mood: Relaxing, Cheerful, Cozy
Theme: Sci-Fi, Space, Nature
Summary: Players settle an interstellar homestead on a verdant exoplanet, piloting a customizable mech to plant and harvest alien

#2
------------------------------------------------------------
Stardew Valley  | score=4.572
Genres: Indie, RPG, Simulation
Native tags: Farming Sim, Pixel Graphics, Multiplayer, Life Sim, RPG
Mood: Relaxing, Cozy, Wholesome
Theme: Slice of Life, Fantasy
Summary: Players inherit an overgrown farm and rebuild it by growing crops, raising animals, crafting, and befriending the townsf

#3
------------------------------------------------------------
Raft  | score=3.065
Genres: Adventure, Indie, Simulatio

In [4]:
import json
import re
from pathlib import Path

from rank_bm25 import BM25Okapi

# ============================================================
# CONFIG
# ============================================================

DATA_DIR = Path(
    "/Users/kesiyun/Desktop/5481 Data/5481Project Data/Steam_data_full"
)

ITEMS_FILE = DATA_DIR / "items.jsonl"
TAGS_FILE = DATA_DIR / "tags.jsonl"

# ============================================================
# HELPERS
# ============================================================

def load_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))

    return rows


def tokenize(text):
    """
    Simple tokenizer for BM25.
    """

    return re.findall(r"\w+", text.lower())


# ============================================================
# LOAD DATA
# ============================================================

print("Loading items...")
items = load_jsonl(ITEMS_FILE)

print("Loading annotations...")
annotations = load_jsonl(TAGS_FILE)

annotation_map = {
    row["item_id"]: row
    for row in annotations
}

item_map = {
    row["item_id"]: row
    for row in items
}

print(f"Items loaded: {len(items):,}")
print(f"Annotations loaded: {len(annotation_map):,}")

# ============================================================
# BM25 WEIGHTS
# ============================================================
'''
NAME_WEIGHT = 5
SUMMARY_WEIGHT = 3
GENRE_WEIGHT = 3
NATIVE_TAG_WEIGHT = 2
LLM_TAG_WEIGHT = 2
DESCRIPTION_WEIGHT = 1
'''
NAME_WEIGHT = 0
SUMMARY_WEIGHT = 1
GENRE_WEIGHT = 0
NATIVE_TAG_WEIGHT = 0
LLM_TAG_WEIGHT = 1
DESCRIPTION_WEIGHT = 1


# ============================================================
# BUILD BM25 CORPUS
# ============================================================

documents = []

for idx, item in enumerate(items):

    item_id = item["item_id"]

    anno = annotation_map.get(item_id, {})

    summary = anno.get("summary", "")

    tag_obj = anno.get("tags", {})

    llm_tokens = []

    for dimension in [
        "genre",
        "theme",
        "mechanics",
        "art_style",
        "mood",
        "audience",
        "content_warnings"
    ]:

        values = tag_obj.get(dimension, [])

        if values:
            llm_tokens.extend(values)

    # --------------------------------------------------------
    # Weighted fields
    # --------------------------------------------------------

    name_text = (
        (item.get("name", "") + " ")
        * NAME_WEIGHT
    ) 

    summary_text = (
        (summary + " ")
        * SUMMARY_WEIGHT
    ) # tags.jsonl - summary, LLM generated

    genre_text = (
        (" ".join(item.get("genres", [])) + " ")
        * GENRE_WEIGHT
    ) # platform categories

    native_tag_text = (
        (" ".join(item.get("native_tags", [])) + " ")
        * NATIVE_TAG_WEIGHT
    ) # users tags

    llm_tag_text = (
        (" ".join(llm_tokens) + " ")
        * LLM_TAG_WEIGHT
    ) #packed from dimension in tags.jsonl - tags, LLM generated the tags from description.

    description_text = (
        (item.get("description_clean", "") + " ")
        * DESCRIPTION_WEIGHT
    ) # from item.jsonl. original (while cleaned), not LLM output.

    # --------------------------------------------------------
    # Final retrievable document
    # --------------------------------------------------------

    searchable_text = " ".join([
        name_text,
        summary_text,
        genre_text,
        native_tag_text,
        llm_tag_text,
        description_text
    ])

    documents.append(
        tokenize(searchable_text)
    )

    if (idx + 1) % 5000 == 0:
        print(
            f"Processed {idx+1:,} documents"
        )

print("Building BM25 index...")
bm25 = BM25Okapi(documents)
print("BM25 ready.")

# ============================================================
# FREE TEXT SEARCH
# ============================================================

def search(query, top_k=10):

    query_tokens = tokenize(query)

    scores = bm25.get_scores(query_tokens)

    ranked = sorted(
        zip(items, scores),
        key=lambda x: x[1],
        reverse=True
    )

    return ranked[:top_k]


# ============================================================
# ITEM -> ITEM RETRIEVAL
# ============================================================

def recommend_similar(item_id, top_k=10):

    target = item_map[item_id]

    anno = annotation_map.get(item_id, {})

    summary = anno.get("summary", "")

    llm_tokens = []

    for values in anno.get("tags", {}).values():

        if isinstance(values, list):
            llm_tokens.extend(values)

    query_text = " ".join([
        target.get("name", ""),
        summary,
        " ".join(target.get("genres", [])),
        " ".join(target.get("native_tags", [])),
        " ".join(llm_tokens)
    ])

    raw_results = search(query_text, top_k + 1)

    filtered = [
        (game, score)
        for game, score in raw_results
        if game["item_id"] != item_id
    ]

    return filtered[:top_k]


# ============================================================
# PRETTY PRINT
# ============================================================

def print_game(game, score):

    item_id = game["item_id"]

    anno = annotation_map.get(item_id, {})

    print("-" * 80)

    print(
        f"{game['name']} "
        f"(id={item_id}) "
        f"score={score:.3f}"
    )

    print(
        "Genres:",
        ", ".join(game.get("genres", [])[:8])
    )

    print(
        "Native Tags:",
        ", ".join(game.get("native_tags", [])[:8])
    )

    tags = anno.get("tags", {})

    if tags.get("mood"):
        print(
            "Mood:",
            ", ".join(tags["mood"])
        )

    if tags.get("theme"):
        print(
            "Theme:",
            ", ".join(tags["theme"])
        )

    summary = anno.get("summary", "")

    if summary:

        print(
            "Summary:",
            summary[:180]
        )


# ============================================================
# DEMO 1
# ============================================================

print("\n")
print("="*80)
print("FREE TEXT RETRIEVAL")
print("="*80)

#query = "relaxing farming game"
query = "truck driving game"

print(f"Query: {query}")


results = search(query, top_k=10)

for rank, (game, score) in enumerate(results, start=1):

    print(f"\nRank #{rank}")

    print_game(game, score)


# ============================================================
# DEMO 2
# ============================================================

print("\n")
print("="*80)
print("ITEM TO ITEM RETRIEVAL")
print("="*80)

target_id = 413150  # Stardew Valley
print(f"target_id: {target_id}")

target = item_map[target_id]

print(
    f"\nTarget item: {target['name']} ({target_id})"
)

results = recommend_similar(
    target_id,
    top_k=10
)

for rank, (game, score) in enumerate(results, start=1):

    print(f"\nRank #{rank}")

    print_game(game, score)

Loading items...
Loading annotations...
Items loaded: 25,899
Annotations loaded: 25,899
Processed 5,000 documents
Processed 10,000 documents
Processed 15,000 documents
Processed 20,000 documents
Processed 25,000 documents
Building BM25 index...
BM25 ready.


FREE TEXT RETRIEVAL
Query: truck driving game

Rank #1
--------------------------------------------------------------------------------
Transporter Truck Simulator (id=1091570) score=20.585
Genres: Adventure, Casual, Indie, Racing, Simulation, Sports
Native Tags: Racing, Sports, Driving, Survival, Relaxing, Casual, Adventure, Simulation
Mood: Challenging, Other
Theme: Modern
Summary: A truck driving simulator in which the player hauls oil tanks and other cargo along challenging roads, parking rigs and meeting time limits. Career mode earns money for new trucks 

Rank #2
--------------------------------------------------------------------------------
Extreme Truck Stunts (id=2148760) score=18.733
Genres: Casual, Indie, Racing, Simul

In [5]:
# ============================================================
# DEMO 1
# ============================================================

print("\n")
print("="*80)
print("FREE TEXT RETRIEVAL")
print("="*80)

#query = "relaxing farming game"
query = 'A Game About Chopping Trees' #'a truck driving game featured in europe' #"a relaxing video game with beautiful art but no combat" #"truck driving game"

print(f"Query: {query}")


results = search(query, top_k=10)

for rank, (game, score) in enumerate(results, start=1):

    print(f"\nRank #{rank}")

    print_game(game, score)



FREE TEXT RETRIEVAL
Query: A Game About Chopping Trees

Rank #1
--------------------------------------------------------------------------------
A Game About Chopping Trees (id=4512570) score=28.495
Genres: Casual, Simulation
Native Tags: Simulation, Relaxing, Cozy, Exploration, Nature, Atmospheric, Stylized, Casual
Mood: Relaxing, Cozy, Contemplative
Theme: Nature, Slice of Life
Summary: The player lives the quiet life of a woodsman, chopping trees at their own pace, riding a handcar between peaceful forests, spending earnings on sharper axes and stronger swings, a

Rank #2
--------------------------------------------------------------------------------
It's Chopping Time! (id=4272270) score=28.266
Genres: Casual, Indie, Simulation, Strategy
Native Tags: Casual, Incremental, Idler, Strategy, Relaxing, Indie, Simulation, Isometric
Mood: Relaxing, Cheerful
Theme: Nature, Mythology
Summary: An incremental game where hovering the cursor over trees chops them down automatically on a secl

In [11]:
# ============================================================
# DEMO 2
# ============================================================

print("\n")
print("="*80)
print("ITEM TO ITEM RETRIEVAL")
print("="*80)

target_id = 227300 #227300- ets2 #413150  # Stardew Valley
print(f"target_id: {target_id}")

target = item_map[target_id]

print(
    f"\nTarget item: {target['name']} ({target_id})"
)

results = recommend_similar(
    target_id,
    top_k=10
)

for rank, (game, score) in enumerate(results, start=1):

    print(f"\nRank #{rank}")

    print_game(game, score)



ITEM TO ITEM RETRIEVAL
target_id: 227300

Target item: Euro Truck Simulator 2 (227300)

Rank #1
--------------------------------------------------------------------------------
American Truck Simulator (id=270880) score=154.622
Genres: Indie, Simulation
Native Tags: Automobile Sim, Driving, Open World, Simulation, Transportation, Realistic, Moddable, Relaxing
Mood: Relaxing, Atmospheric
Theme: Modern
Summary: Players take the wheel of licensed American trucks, hauling cargo across California, Nevada, and Arizona while building a freight company from hired driver to owner-operator. The s

Rank #2
--------------------------------------------------------------------------------
Motor Town: Behind The Wheel (id=1369670) score=137.588
Genres: Adventure, Casual, Indie, Racing, Simulation, Early Access
Native Tags: Early Access, Driving, Open World, Simulation, Transportation, Automobile Sim, Multiplayer, Relaxing
Mood: Relaxing, Cheerful
Theme: Modern
Summary: Players drive cars, buses, an

In [7]:
import os
import dashscope
dashscope.base_http_api_url = "https://ws-7ja1tvpg0mgc6vna.cn-beijing.maas.aliyuncs.com/api/v1"

messages = [
    {
        "role": "user",
        "content": [
            {"text": "Hi!"}]
    }]
response = dashscope.MultiModalConversation.call(
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    model='qwen3.8-27b',
    messages=messages
)
print(response.output.choices[0].message.content[0]["text"])


Hey there! 👋 How's it going? What can I help you with today?


In [13]:
# ============================================================
# DEMO 3
# FREE TEXT RETRIEVAL + LLM EXPLANATION
# ============================================================

print("\n")
print("="*80)
print("FREE TEXT RETRIEVAL")
print("="*80)

query = "a relaxing video game with beautiful art but no combat"

print(f"Query: {query}")

results = search(query, top_k=10)

# ------------------------------------------------------------
# Generate explanation for TOP-1 result
# ------------------------------------------------------------

top_game, score = results[0]

item_id = top_game["item_id"]
anno = annotation_map[item_id]

SYSTEM_PROMPT = """
You are a game recommendation explanation assistant.

Explain why the retrieved game matches the user's query.

Rules:
- Maximum 60 words.
- Friendly and natural.
- Focus on gameplay, theme and mood.
- Do not mention BM25, search engines or ranking.
"""

USER_PROMPT = f"""
User query:
{query}

Retrieved game:
{top_game['name']}

Summary:
{anno.get('summary','')}

Tags:
{json.dumps(anno.get('tags', {}), ensure_ascii=False)}
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": [{"text": USER_PROMPT}]}
]

response = dashscope.MultiModalConversation.call(
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    model='qwen3.8-27b',
    messages=messages
)

explanation = response.output.choices[0].message.content[0]["text"]

print("\n")
print("LLM Explanation")
print("-"*80)
print(explanation)

# ------------------------------------------------------------
# Print retrieved results
# ------------------------------------------------------------

for rank, (game, score) in enumerate(results, start=1):

    print(f"\nRank #{rank}")

    print_game(game, score)



FREE TEXT RETRIEVAL
Query: a relaxing video game with beautiful art but no combat


LLM Explanation
--------------------------------------------------------------------------------
Pleasure Puzzle: Portrait offers a calm, cozy jigsaw experience with beautiful realistic portraits. You slowly piece together artworks to soothing music, making it perfect if you want relaxing gameplay without any combat. It’s a gentle, creative puzzle game focused on appreciation and completion.

Rank #1
--------------------------------------------------------------------------------
Pleasure Puzzle:Portrait 趣拼拼：肖像画 (id=939620) score=26.940
Genres: Casual, Indie, Simulation
Native Tags: Colorful, Tabletop, Time Management, Board Game, Point & Click, Music, Singleplayer, Casual
Mood: Relaxing, Cozy
Theme: Other
Summary: A classic jigsaw puzzle game built from one hundred authorized portrait images, with custom difficulty on the first hundred puzzles. Players assemble pieces while listening to back

Rank #2

In [9]:
# ============================================================
# DEMO 4
# ITEM -> ITEM RETRIEVAL + LLM EXPLANATION
# ============================================================

print("\n")
print("="*80)
print("ITEM TO ITEM RETRIEVAL")
print("="*80)

target_id = 227300

target = item_map[target_id]

print(f"target_id: {target_id}")

print(
    f"\nTarget item: {target['name']} ({target_id})"
)

results = recommend_similar(
    target_id,
    top_k=10
)

# ------------------------------------------------------------
# Explain Top-1 recommendation
# ------------------------------------------------------------

recommended_game, score = results[0]

source_anno = annotation_map[target_id]
recommended_anno = annotation_map[
    recommended_game["item_id"]
]

SYSTEM_PROMPT = """
You are a game recommendation explanation assistant.

Explain why the recommended game is similar to the source game.

Rules:
- Maximum 80 words.
- Mention gameplay, themes or player experience.
- Do not mention BM25, ranking or search algorithms.
"""

USER_PROMPT = f"""
Source Game:
{target['name']}

Source Summary:
{source_anno.get('summary','')}

Source Tags:
{json.dumps(source_anno.get('tags', {}), ensure_ascii=False)}

Recommended Game:
{recommended_game['name']}

Recommended Summary:
{recommended_anno.get('summary','')}

Recommended Tags:
{json.dumps(recommended_anno.get('tags', {}), ensure_ascii=False)}
"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": [{"text": USER_PROMPT}]}
]

response = dashscope.MultiModalConversation.call(
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    model='qwen3.8-27b',
    messages=messages
)

explanation = response.output.choices[0].message.content[0]["text"]

print("\n")
print("LLM Explanation")
print("-"*80)
print(explanation)

# ------------------------------------------------------------
# Show recommendations
# ------------------------------------------------------------

for rank, (game, score) in enumerate(results, start=1):

    print(f"\nRank #{rank}")

    print_game(game, score)



ITEM TO ITEM RETRIEVAL
target_id: 227300

Target item: Euro Truck Simulator 2 (227300)


LLM Explanation
--------------------------------------------------------------------------------
Both games offer realistic first-person truck driving and management gameplay. Players haul cargo across expansive, real-world road networks, customize trucks, and grow a freight business by buying vehicles, hiring drivers, and managing routes. The calm, contemplative mood and detailed simulation suit players who enjoy Euro Truck Simulator 2’s slow-paced, immersive driving and company-building experience.

Rank #1
--------------------------------------------------------------------------------
American Truck Simulator (id=270880) score=154.622
Genres: Indie, Simulation
Native Tags: Automobile Sim, Driving, Open World, Simulation, Transportation, Realistic, Moddable, Relaxing
Mood: Relaxing, Atmospheric
Theme: Modern
Summary: Players take the wheel of licensed American trucks, hauling cargo across Cali